In [1]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.append(os.getcwd())
print(os.getcwd())

C:\Users\alvee\Documents\ai-grid-fault-classifier


In [2]:
%%writefile common.py
import pandas as pd

TARGET = "fault_types"
FEATURES = ["Ia","Ib","Ic","Va","Vb","Vc","Im","Vm","Pai","Pav","Fi","Fv"]
CLASSES = ["NF","LG","LL","LLG","LLL","LLLG"]
RANDOM_STATE = 42

def load_clean(path="data/processed/clean.csv"):
    return pd.read_csv(path)

def blocked_split(df, n_blocks=5):
    df = df.copy()
    n = df.groupby(TARGET)[TARGET].transform("size")
    df["block"] = df.groupby(TARGET).cumcount() * n_blocks // n
    train = df[df["block"] < n_blocks - 1]   # first 80% of every run
    test = df[df["block"] == n_blocks - 1]   # last 20% of every run
    return train, test

Writing common.py


In [3]:
from common import *

df = pd.read_csv("data/raw/PMU_Fault_Data.csv")

In [4]:
df.head()

,time_stamp,Ia,Ib,Ic,Va,Vb,Vc,Im,Vm,Pai,Pav,Fi,Fv,fault_types
0,0.000000e+00,1963.798615,-8897.985690,6934.187075,64.010540,-285.098287,221.087747,1.0,1.0,0.0,0.0,50.0,50.0,NF
1,3.160000e-30,1963.798615,-8897.985690,6934.187075,64.010540,-285.098287,221.087747,1.0,1.0,0.0,0.0,50.0,50.0,NF
2,3.125000e-04,2849.984363,-9136.332309,6286.347925,92.337771,-292.611899,200.274127,1.0,1.0,0.0,0.0,50.0,50.0,NF
3,6.250000e-04,3708.810435,-9286.742028,5577.931566,119.778538,-297.309131,177.530592,1.0,1.0,0.0,0.0,50.0,50.0,NF
4,9.375000e-04,4531.982598,-9347.752715,4815.770110,146.067826,-299.144311,153.076485,1.0,1.0,0.0,0.0,50.0,50.0,NF


In [5]:
print(df.shape, df.isna().sum().sum())      # (134406, 14) 0
print(df[TARGET].value_counts())            # 22,401 per class

df = df.drop_duplicates()                   # removes 17 rows
warmup = (df["Im"] == 1.0) & (df["Vm"] == 1.0)
df = df[~warmup].copy()                     # removes 370 rows
df = df.sort_values([TARGET, "time_stamp"], kind="stable").reset_index(drop=True)
df.to_csv("data/processed/clean.csv", index=False)
print(df.shape)                             # (134019, 14)

train, test = blocked_split(df)
print(len(train), len(test))                

(134406, 14) 0
fault_types
NF      22401
LG      22401
LL      22401
LLG     22401
LLL     22401
LLLG    22401
Name: count, dtype: int64
(134019, 14)
107218 26801
